# 预测未来 Occupancy

驾驶路线不以‘下一帧好看’为主要目标。我们把过去三帧俯视占用与候选 ego action 变成未来三帧占用。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.spatial import TinyOccupancyPredictor, make_moving_occupancy_dataset, occupancy_iou
torch.manual_seed(1)


## 1. 历史、计划与未来必须分开

过去三帧说明物体怎样运动；action 表示接下来选择的控制；未来三帧是监督目标。

In [ ]:
history, actions, future = make_moving_occupancy_dataset(96, seed=1)
print('history/action/future:', tuple(history.shape), tuple(actions.shape), tuple(future.shape))
assert history.shape == future.shape == (96, 3, 16, 16)


## 2. 动作条件未来占用

动作 embedding 被铺到整个 BEV，再与历史帧一起交给卷积网络。输出每个未来格子的 occupied logit。

In [ ]:
model = TinyOccupancyPredictor()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
positive_weight = torch.tensor(18.0)
losses = []
for _ in range(80):
    opt.zero_grad(); logits = model(history, actions); loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, future, pos_weight=positive_weight); loss.backward(); opt.step(); losses.append(float(loss.detach()))
iou = occupancy_iou(logits.detach(), future)
print('loss:', round(losses[0], 3), '→', round(losses[-1], 3), 'IoU:', round(float(iou), 3))
assert losses[-1] < losses[0]


## 3. 同一历史替换计划

如果数据真的包含 action-conditioned future，同一历史换动作后，未来 Occupancy 应改变。没有 ego action 的数据只能训练 open-loop future predictor。

In [ ]:
same_history = history[:1].expand(5, -1, -1, -1)
all_actions = torch.arange(5)
with torch.no_grad(): counterfactual = torch.sigmoid(model(same_history, all_actions))
differences = [(counterfactual[0] - counterfactual[i]).abs().mean().item() for i in range(1, 5)]
print('换动作后的 occupancy 差异:', [round(x, 4) for x in differences])
assert max(differences) > 0


## 4. Open-loop 还不是自动驾驶闭环

IoU 只检查离线占用。碰撞率、驶出道路率和安全接管需要 Planner、车辆动力学和可交互模拟器；本 Notebook 不报告这些指标。

In [ ]:
print('本实验可以报告: future occupancy IoU、horizon 曲线、动作敏感性')
print('本实验不能报告: 闭环碰撞率、自动驾驶成功率')


## 小结

这一份把空间动态的输出固定为未来 Occupancy。8.8 的驾驶分支再加入多相机 LSS、nuScenes-mini 和严格的数据边界；没有控制或模拟器时仍只称 open-loop predictor。